In [15]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd

In [18]:
train_df = pd.read_csv('../data/train.csv')
train_df.head()

# def load_data():
#     """Load few-shot training data for SATD detection."""
#     samples = [
#         {"text": "TODO: Refactor this function", "label": 1},
#         {"text": "This needs to be optimized", "label": 1},
#         {"text": "Fix this later", "label": 1},
#         {"text": "Implemented as per requirement", "label": 0},
#         {"text": "Adding a simple function", "label": 0},
#         {"text": "Refactor and improve code", "label": 1},
#         {"text": "Optimize performance for large dataset", "label": 1},
#         {"text": "Simple function without any known issues", "label": 0},
#         {"text": "This needs some cleanup", "label": 1},
#         {"text": "Completed feature as expected", "label": 0},
#     ]
#     return Dataset.from_dict({"text": [s["text"] for s in samples], "label": [s["label"] for s in samples]})

x = []
y = []
for index, row in train_df.head(10).iterrows():
    x.append(row['text'])
    y.append(1)
    

training_samples = Dataset.from_dict({"text": x, "label":y})


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Check if CUDA (GPU) is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def tokenize_function(example, tokenizer):
    return tokenizer(example["text"], padding=True, truncation=True, max_length=512)

def compute_metrics(p):
    """Compute metrics like accuracy, precision, recall, and F1 score."""
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='binary')
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

def train_model(model_name="roberta-base", num_epochs=3, dataset):
    """Train SATD detection model with a specified transformer model."""
    dataset = load_data()
    dataset = dataset.train_test_split(test_size=0.2)  # Split into training and testing
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(device)  # Ensure the model is moved to the correct device (GPU or CPU)
    
    training_args = TrainingArguments(
        output_dir="./cache/results",
        num_train_epochs=num_epochs,
        eval_strategy="epoch",  # Updated from `evaluation_strategy` to `eval_strategy`
        save_strategy="epoch",  # Matching evaluation_strategy
        load_best_model_at_end=True,
        logging_dir="./cache/logs",
        logging_steps=10,             # Log every 10 steps
        per_device_train_batch_size=2,  # Smaller batch size for few-shot
        per_device_eval_batch_size=2,
        warmup_steps=500,
        weight_decay=0.01,
        report_to="none"  # Prevent reporting to external services like WandB
    )
    
    trainer = Trainer(
        model=model, 
        args=training_args, 
        train_dataset=dataset['train'], 
        eval_dataset=dataset['test'],
        tokenizer=tokenizer,  # Updated to tokenizer=tokenizer, as this will be deprecated in future versions
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    return model, tokenizer, dataset

def predict(model, tokenizer, texts):
    """Make predictions using the trained model."""
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)  # Move inputs to the same device
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()  # Move outputs back to CPU for NumPy
    return predictions

# Example usage
if __name__ == "__main__":
    model_name = "roberta-base"  # Change this to switch models (e.g., "bert-base-uncased", "distilbert-base-uncased", etc.)
    model, tokenizer, dataset = train_model(model_name)

    # Test predictions on the test set only
    test_samples = dataset['test']['text']
    test_predictions = predict(model, tokenizer, test_samples)
    
    test_labels = dataset['test']['label']
    test_accuracy = accuracy_score(test_labels, test_predictions)

    print(f"Test Accuracy: {test_accuracy}")

    # Compute precision, recall, and F1 score for the test set
    test_metrics = precision_recall_fscore_support(test_labels, test_predictions, average='binary')

    print(f"Test Precision, Recall, F1: {test_metrics}")


Map: 100%|██████████| 2/2 [00:00<00:00, 639.38 examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-13-e562216e1cec>:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.555351,1.000000,1.000000,1.000000,1.000000
2,No log,0.555862,1.000000,1.000000,1.000000,1.000000
3,0.728800,0.556947,1.000000,1.000000,1.000000,1.000000


Test Accuracy: 1.0
Test Precision, Recall, F1: (1.0, 1.0, 1.0, None)
